# ⏳ Phase 2 — Reliable Waiting & Error Handling
### PixelCraft Infotech | Web Scraping Course

**What you'll learn:**
- Why `time.sleep()` is bad — and what to use instead
- Explicit waits with `WebDriverWait` + `expected_conditions`
- Handling `TimeoutException` and `NoSuchElementException`
- Retry logic for flaky pages
- Recovering from `StaleElementReferenceException`

---

## Step 1 — The Problem with time.sleep()

> `time.sleep(2)` always waits 2 seconds — even if the page loaded in 0.3s.  
> On a slow connection it might not wait *long enough* — causing errors.  
> **Explicit waits** poll until the element appears — fast when fast, patient when slow.

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException
)
import time

print("✅ Imports done")

✅ Imports done


---
## Step 2 — WebDriverWait + expected_conditions

> `WebDriverWait(driver, 10)` — wait UP TO 10 seconds  
> `.until(EC.presence_of_element_located(...))` — stop as soon as element appears

In [2]:
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js")

# ❌ Old way — dumb wait
# time.sleep(2)

# ✅ New way — smart wait
wait = WebDriverWait(driver, 10)   # max 10 seconds

# Wait until at least one quote appears on the page
first_quote = wait.until(
    EC.presence_of_element_located((By.CLASS_NAME, "quote"))
)

print("✅ Page loaded — first quote found!")
print(first_quote.text[:100])
driver.quit()

✅ Page loaded — first quote found!
“The world as we have created it is a process of our thinking. It cannot be changed without changing


---
## Step 3 — Common Expected Conditions

| Condition | Use when... |
|---|---|
| `presence_of_element_located` | Element exists in DOM (may be hidden) |
| `visibility_of_element_located` | Element is visible on screen |
| `element_to_be_clickable` | Element is visible AND clickable |
| `presence_of_all_elements_located` | All matching elements loaded |
| `text_to_be_present_in_element` | Wait for specific text to appear |

In [3]:
driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js")

wait = WebDriverWait(driver, 10)

# Wait for ALL quotes to be present
all_quote_elements = wait.until(
    EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
)
print(f"Found {len(all_quote_elements)} quotes")

# Wait for next button to be clickable before clicking
next_btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a"))
)
print("Next button is clickable — clicking...")
next_btn.click()

# Wait for page 2 quotes to load
wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
print("Page 2 loaded:", driver.current_url)

driver.quit()

Found 10 quotes
Next button is clickable — clicking...
Page 2 loaded: https://quotes.toscrape.com/js/page/2/


---
## Step 4 — Handling TimeoutException

> If the element doesn't appear within the timeout, Selenium raises `TimeoutException`.  
> Always wrap waits in `try/except` to handle this gracefully.

In [4]:
driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js")

wait = WebDriverWait(driver, 10)

try:
    # Try to find an element that doesn't exist
    ghost_element = wait.until(
        EC.presence_of_element_located((By.ID, "this-does-not-exist"))
    )
except TimeoutException:
    print("⏰ TimeoutException: Element not found within 10 seconds")
    print("Continuing with the rest of the script...")

# Script continues normally
quotes = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
print(f"✅ Found {len(quotes)} quotes anyway")

driver.quit()

⏰ TimeoutException: Element not found within 10 seconds
Continuing with the rest of the script...
✅ Found 10 quotes anyway


---
## Step 5 — Handling NoSuchElementException

> This happens when you use `find_element` (not wait) and the element isn't there.  
> Common case: checking if the 'Next' button exists

In [ ]:
def safe_find(driver, by, selector, default=None):
    """Find an element, return default if not found."""
    try:
        return driver.find_element(by, selector)
    except NoSuchElementException:
        return default


driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js/page/10/")  # Last page — no Next button

wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))

next_btn = safe_find(driver, By.CSS_SELECTOR, "li.next a")

if next_btn:
    next_btn.click()
    print("Clicked next")
else:
    print("🛑 No Next button — this is the last page")

driver.quit()

---
## Step 6 — StaleElementReferenceException

> Happens when the DOM refreshes AFTER you've found an element but BEFORE you use it.  
> The element reference becomes 'stale' (outdated).  
> **Fix:** Re-find the element after any page change.

In [ ]:
def click_with_retry(driver, css_selector, retries=3):
    """Click an element, retry if stale."""
    for attempt in range(retries):
        try:
            wait = WebDriverWait(driver, 10)
            btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, css_selector)))
            btn.click()
            return True
        except StaleElementReferenceException:
            print(f"⚠️ Stale element on attempt {attempt+1}, retrying...")
            time.sleep(0.5)
    return False


driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js")

wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))

success = click_with_retry(driver, "li.next a")
if success:
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))
    print("✅ Navigated to:", driver.current_url)

driver.quit()

---
## Step 7 — Full Robust Scraper (Putting it all together)

> This is how a **production-grade** scraper looks — no `time.sleep`, proper error handling

In [ ]:
import csv

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js")

wait = WebDriverWait(driver, 15)   # generous timeout
all_quotes = []
page = 1

while True:
    print(f"Scraping page {page}...")
    try:
        # Wait for quotes to load
        quote_els = wait.until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
        )

        for q in quote_els:
            try:
                text   = q.find_element(By.CLASS_NAME, "text").text.strip()
                author = q.find_element(By.CLASS_NAME, "author").text.strip()
                tags   = [t.text for t in q.find_elements(By.CLASS_NAME, "tag")]
                all_quotes.append({
                    "quote": text,
                    "author": author,
                    "tags": ", ".join(tags),
                    "page": page
                })
            except StaleElementReferenceException:
                print(f"  ⚠️ Stale element skipped on page {page}")

        print(f"  ✅ {len(quote_els)} quotes scraped")

        # Try to click Next
        try:
            next_btn = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a"))
            )
            next_btn.click()
            page += 1

        except TimeoutException:
            print("  🛑 No Next button — done!")
            break

    except TimeoutException:
        print(f"  ❌ Page {page} timed out — skipping")
        break

driver.quit()
print(f"\n✅ Total scraped: {len(all_quotes)} quotes")

# Save to CSV
with open("robust_quotes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["quote", "author", "tags", "page"])
    writer.writeheader()
    writer.writerows(all_quotes)

print("✅ Saved to robust_quotes.csv")

---
## 🏆 Class Exercise

1. Replace all `time.sleep()` in your Phase 1 notebook with proper `WebDriverWait`
2. Add a `safe_find()` function and use it for the Next button check
3. Wrap the entire scrape loop in a `try/except TimeoutException`
4. Test it on a slow page — add `driver.set_page_load_timeout(20)` at the top
5. Print a message for each exception caught — don't let the script crash silently

**Bonus:** Add a retry decorator that re-runs a function up to 3 times on any Selenium exception

In [ ]:
# Bonus: Retry decorator
import functools

def retry_on_exception(retries=3, delay=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt}/{retries} failed: {e}")
                    if attempt < retries:
                        time.sleep(delay)
            raise Exception(f"All {retries} retries failed for {func.__name__}")
        return wrapper
    return decorator


@retry_on_exception(retries=3, delay=1)
def scrape_page(driver, wait):
    return wait.until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
    )

print("✅ Retry decorator ready — decorate any scraping function with @retry_on_exception")